In [54]:
import pandas as pd 
import numpy as np
import random
from datetime import datetime, timedelta

import os
import re
import time
from openai import OpenAI

In [55]:
# create articial dataframe to test narrative extraction -- later replace with real data

# -----------------------------
# 1. Define possible values
# -----------------------------

firms = [
    ("Apple Inc.", "AAPL"),
    ("Microsoft Corp.", "MSFT"),
    ("Tesla Inc.", "TSLA"),
    ("Amazon.com Inc.", "AMZN"),
    ("NVIDIA Corp.", "NVDA"),
    ("Meta Platforms Inc.", "META"),
    ("Alphabet Inc.", "GOOGL"),
]

topics = [
    "Technology",
    "Inflation",
    "Interest Rates",
    "War",
    "Sports",
    "Supply Chains",
    "Energy",
    "China"
]

signs = ["positive", "negative"]


# -----------------------------
# 2. Helper: random date range
# -----------------------------

def random_date_pair(start_year=2018, end_year=2019, max_gap=60):

    start_date = datetime(
        year=random.randint(start_year, end_year),
        month=random.randint(1, 12),
        day=random.randint(1, 28) 
    )

    gap = random.randint(1, max_gap)

    end_date = start_date + timedelta(days=gap)

    return start_date, end_date


# -----------------------------
# 3. Generate DataFrame
# -----------------------------

def generate_random_df(n_rows=3, seed=42):

    random.seed(seed)
    np.random.seed(seed)

    rows = []

    for _ in range(n_rows):

        firm, ticker = random.choice(firms)
        topic = random.choice(topics)
        sign = random.choice(signs)

        start_date, end_date = random_date_pair()

        rows.append({
            "firm_name": firm,
            "ticker": ticker,
            "start_date": start_date,
            "end_date": end_date,
            "topic": topic,
            "sign": sign
        })

    df = pd.DataFrame(rows)

    return df

data = generate_random_df(n_rows=5)



In [59]:
# now we create prompts from the articial data

client = OpenAI()   # assumes OPENAI_API_KEY is set


# -----------------------------
# 1) Prompt builder
# -----------------------------

def build_prompt(row: pd.Series) -> str:
    firm = row["firm_name"]
    ticker = row["ticker"]
    start_date = pd.to_datetime(row["start_date"]).date()
    end_date = pd.to_datetime(row["end_date"]).date()
    topic = row["topic"]
    sign = row["sign"]

    return (
        f'Could "{topic}" be a predictor of the returns of {firm} ({ticker}) '
        f'with a {sign} sign between {start_date} and {end_date}? '
        "Was there a specific event that makes this likely, or is this topic irrelevant?\n\n"
        "Search your background knowledge of the time period.\n\n"
        "Return EXACTLY one of the following:\n"
        "1) NA\n"
        "2) a factual description of the relevant event(s)\n"
        "No extra text, no bullet points."
    )


# -----------------------------
# 2) API Call (no truncation)
# -----------------------------

def query_openai(prompt: str,
                 model: str = "gpt-4o-mini",
                 max_retries: int = 4) -> str:
    """
    Uses the Responses API.
    Returns raw model output (unlimited length) or 'NA' on failure.
    """

    for attempt in range(max_retries):
        try:
            resp = client.responses.create(
                model=model,
                input=prompt,
            )

            out = getattr(resp, "output_text", "").strip()

            if not out:
                return "NA"

            return out

        except Exception:
            if attempt == max_retries - 1:
                return "NA"

            time.sleep(2 ** attempt)


# -----------------------------
# 3) Apply to DataFrame
# -----------------------------

def add_llm_column(df: pd.DataFrame,
                   model: str = "gpt-4o-mini") -> pd.DataFrame:

    df = df.copy()

    df["prompt"] = df.apply(build_prompt, axis=1)

    df["llm_response"] = df["prompt"].apply(
        lambda p: query_openai(p, model=model)
    )

    return df

In [60]:
df = add_llm_column(data, model="gpt-4o-mini")

In [61]:
# loop over and print each row in df
for index, row in df.iterrows():
    print(f"Row {index}:")
    print(f"Firm: {row['firm_name']} ({row['ticker']})")
    print(f"Date Range: {row['start_date'].date()} to {row['end_date'].date()}")
    print(f"Topic: {row['topic']}")
    print(f"Sign: {row['sign']}")
    print(f"LLM Response: {row['llm_response']}")
    print("-" * 40)

Row 0:
Firm: Meta Platforms Inc. (META)
Date Range: 2019-04-08 to 2019-04-17
Topic: Inflation
Sign: positive
LLM Response: 2) A factual description of the relevant event(s)
----------------------------------------
Row 1:
Firm: Meta Platforms Inc. (META)
Date Range: 2019-01-01 to 2019-01-07
Topic: Inflation
Sign: positive
LLM Response: NA
----------------------------------------
Row 2:
Firm: Microsoft Corp. (MSFT)
Date Range: 2018-12-21 to 2019-02-04
Topic: War
Sign: positive
LLM Response: During the specified period from 2018-12-21 to 2019-02-04, the U.S.-China trade tensions escalated, particularly regarding tariffs and technology. These tensions created uncertainties in global markets, impacting technology stocks like Microsoft. However, the potential for resolution in trade negotiations around that time caused some fluctuations in market sentiment. Therefore, the relevance of war or military conflict as a predictor specifically for Microsoft's returns is limited; the focus was prima